In [1]:
#This space is to note or download any libraries , modules as required

In [2]:
"""
Upgraded LightGBM pipeline: Feature engineering + per-biller error + randomized hyperparameter tuning
File (input): /mnt/data/daily_all_billers2_expanded_with_billers.csv
Outputs:
 - lightgbm_val_results_both_tuned.csv     (validation predictions)
 - biller_level_error_report.csv          (per-biller error metrics)
 - best_params_transactions.json
 - best_params_amount.json
 - model_transactions.txt                 (LightGBM model)
 - model_amount.txt                       (LightGBM model)
 - optionally feature importance & plot windows shown inline
"""

'\nUpgraded LightGBM pipeline: Feature engineering + per-biller error + randomized hyperparameter tuning\nFile (input): /mnt/data/daily_all_billers2_expanded_with_billers.csv\nOutputs:\n - lightgbm_val_results_both_tuned.csv     (validation predictions)\n - biller_level_error_report.csv          (per-biller error metrics)\n - best_params_transactions.json\n - best_params_amount.json\n - model_transactions.txt                 (LightGBM model)\n - model_amount.txt                       (LightGBM model)\n - optionally feature importance & plot windows shown inline\n'

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
import json, os, random, joblib, warnings
from datetime import timedelta
from math import sqrt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings("ignore")
random_seed = 42
np.random.seed(random_seed)
random.seed(random_seed)

In [4]:
# ---------- PARAMETERS ----------
CSV_PATH = "daily_all_billers2_expanded_with_billers.csv"
OUT_PRED_CSV = "lightgbm_val_results_both_tuned_Take_1.csv"
OUT_BILLER_ERR = "biller_level_error_report_Take_1.csv"
BEST_PARAMS_TXN = "best_params_transactions_Take_1.json"
BEST_PARAMS_AMT = "best_params_amount_Take_1.json"
MODEL_TXN_PATH = "model_transactions_Take_1.txt"
MODEL_AMT_PATH = "model_amount_Take_1.txt"
VAL_DAYS = 28
# Randomized tuning trials (increase if you have more time/CPU)
N_TRIALS = 25

In [5]:
# Columns (as you provided)
DATE_COL = "txn_date"
BILLER_COL = "biller_id"
TARGET_TXN = "total_transactions"
TARGET_AMT = "total_amount"


In [6]:
# ---------- HELPERS ----------
def safe_mape(y_true, y_pred):
    denom = np.clip(y_true, 1e-6, None)
    return np.mean(np.abs((y_true - y_pred) / denom)) * 100

def metrics_reg(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = sqrt(mean_squared_error(y_true, y_pred))
    mape = safe_mape(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {'MAE': mae, 'RMSE': rmse, 'MAPE(%)': mape, 'R2': r2}

In [7]:
# ---------- LOAD ----------
print("Loading:", CSV_PATH)
df = pd.read_csv(CSV_PATH)
print("Initial shape:", df.shape)
# parse date
df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce')
df = df.dropna(subset=[DATE_COL]).sort_values([BILLER_COL, DATE_COL]).reset_index(drop=True)

Loading: daily_all_billers2_expanded_with_billers.csv
Initial shape: (10000000, 5)


In [8]:
'''# Aggregate if duplicates exist per biller-date
multi = df.groupby([BILLER_COL, DATE_COL]).size().reset_index(name='nrows')
if multi['nrows'].max() > 1:
    print("Aggregating multiple rows per (biller, date) by summing targets.")
    other_cols = [c for c in df.columns if c not in [BILLER_COL, DATE_COL, TARGET_TXN, TARGET_AMT]]
    agg_dict = {TARGET_TXN: 'sum', TARGET_AMT: 'sum'}
    for c in other_cols:
        agg_dict[c] = 'first'
    df = df.groupby([BILLER_COL, DATE_COL], as_index=False).agg(agg_dict).sort_values([BILLER_COL, DATE_COL]).reset_index(drop=True)'''

'# Aggregate if duplicates exist per biller-date\nmulti = df.groupby([BILLER_COL, DATE_COL]).size().reset_index(name=\'nrows\')\nif multi[\'nrows\'].max() > 1:\n    print("Aggregating multiple rows per (biller, date) by summing targets.")\n    other_cols = [c for c in df.columns if c not in [BILLER_COL, DATE_COL, TARGET_TXN, TARGET_AMT]]\n    agg_dict = {TARGET_TXN: \'sum\', TARGET_AMT: \'sum\'}\n    for c in other_cols:\n        agg_dict[c] = \'first\'\n    df = df.groupby([BILLER_COL, DATE_COL], as_index=False).agg(agg_dict).sort_values([BILLER_COL, DATE_COL]).reset_index(drop=True)'

In [9]:
# ---------- FEATURE ENGINEERING ----------

# Time features
df['day'] = df[DATE_COL].dt.day
df['dayofweek'] = df[DATE_COL].dt.dayofweek
df['weekofyear'] = df[DATE_COL].dt.isocalendar().week.astype(int)
df['month'] = df[DATE_COL].dt.month
df['is_weekend'] = df['dayofweek'].isin([5,6]).astype(int)
df['is_month_start'] = df[DATE_COL].dt.is_month_start.astype(int)
df['is_month_end'] = df[DATE_COL].dt.is_month_end.astype(int)

In [10]:
# Biller-level historical stats (global)
biller_stats = df.groupby(BILLER_COL).agg(
    biller_txn_mean = (TARGET_TXN, 'mean'),
    biller_txn_std  = (TARGET_TXN, 'std'),
    biller_amt_mean = (TARGET_AMT, 'mean'),
    biller_amt_std  = (TARGET_AMT, 'std'),
    biller_count    = (TARGET_TXN, 'count')
).reset_index().fillna(0)
df = df.merge(biller_stats, on=BILLER_COL, how='left')

In [11]:
# Create extensive lags & rolling features per biller
LAGS = [1,2,3,7,14,21,28,30]
ROLLS = [3,7,14,30]

In [12]:
frames = []
for biller, g in df.groupby(BILLER_COL):
    g = g.sort_values(DATE_COL).copy()
    # lags for txn and amt
    for lag in LAGS:
        g[f'lag_txn_{lag}'] = g[TARGET_TXN].shift(lag)
        g[f'lag_amt_{lag}'] = g[TARGET_AMT].shift(lag)
    # rolling means, medians, stds
    for w in ROLLS:
        g[f'roll_txn_mean_{w}'] = g[TARGET_TXN].shift(1).rolling(window=w, min_periods=1).mean()
        g[f'roll_txn_median_{w}'] = g[TARGET_TXN].shift(1).rolling(window=w, min_periods=1).median()
        g[f'roll_txn_std_{w}'] = g[TARGET_TXN].shift(1).rolling(window=w, min_periods=1).std().fillna(0)
        g[f'roll_amt_mean_{w}'] = g[TARGET_AMT].shift(1).rolling(window=w, min_periods=1).mean()
        g[f'roll_amt_median_{w}'] = g[TARGET_AMT].shift(1).rolling(window=w, min_periods=1).median()
        g[f'roll_amt_std_{w}'] = g[TARGET_AMT].shift(1).rolling(window=w, min_periods=1).std().fillna(0)
    # pct changes and differences
    g['pct_change_1'] = g[TARGET_TXN].pct_change(periods=1).fillna(0)
    g['pct_change_7'] = g[TARGET_TXN].pct_change(periods=7).fillna(0)
    g['diff_1'] = g[TARGET_TXN].diff(periods=1).fillna(0)
    g['diff_7'] = g[TARGET_TXN].diff(periods=7).fillna(0)
    # biller-normalized features
    g['txn_div_biller_mean'] = g[TARGET_TXN] / (g['biller_txn_mean'].replace(0, 1))
    g['txn_minus_biller_mean'] = g[TARGET_TXN] - g['biller_txn_mean']
    g['amt_div_biller_mean'] = g[TARGET_AMT] / (g['biller_amt_mean'].replace(0, 1))
    g['amt_minus_biller_mean'] = g[TARGET_AMT] - g['biller_amt_mean']
    frames.append(g)

In [13]:
df = pd.concat(frames, axis=0).sort_values([BILLER_COL, DATE_COL]).reset_index(drop=True)
df.replace([np.inf, -np.inf], np.nan, inplace=True)


In [14]:
# Drop rows without enough history: require a set of core lags (choose lag 1 and lag 7)
required_lags = [f'lag_txn_{l}' for l in [1,7]] + [f'lag_amt_{l}' for l in [1,7]]
df = df.dropna(subset=required_lags, how='any').reset_index(drop=True)


In [15]:
# Convert biller to categorical
df[BILLER_COL] = df[BILLER_COL].astype('category')

In [16]:
# ---------- TRAIN / VAL SPLIT ----------
max_date = df[DATE_COL].max()
val_start = max_date - timedelta(days=VAL_DAYS-1)
print("Max date:", max_date.date(), "Validation start:", val_start.date())


Max date: 2025-12-30 Validation start: 2025-12-03


In [17]:
df['month'].unique()##just for clarification

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12], dtype=int64)

In [18]:
train = df[df[DATE_COL] < val_start].copy()
val = df[df[DATE_COL] >= val_start].copy()
print("Train rows:", train.shape[0], "Val rows:", val.shape[0])


Train rows: 9961450 Val rows: 38480


In [19]:
# ---------- COMMON FEATURE LIST ----------
# Build feature list programmatically
feat_base = ['day','dayofweek','weekofyear','month','is_weekend','is_month_start','is_month_end']
feat_lags_txn = [f'lag_txn_{l}' for l in LAGS]
feat_rolls_txn = [f'roll_txn_mean_{w}' for w in ROLLS] + [f'roll_txn_median_{w}' for w in ROLLS] + [f'roll_txn_std_{w}' for w in ROLLS]
feat_lags_amt = [f'lag_amt_{l}' for l in LAGS]
feat_rolls_amt = [f'roll_amt_mean_{w}' for w in ROLLS] + [f'roll_amt_median_{w}' for w in ROLLS] + [f'roll_amt_std_{w}' for w in ROLLS]
feat_misc = ['pct_change_1','pct_change_7','diff_1','diff_7','txn_div_biller_mean','txn_minus_biller_mean','amt_div_biller_mean','amt_minus_biller_mean',
             'biller_txn_mean','biller_txn_std','biller_amt_mean','biller_amt_std','biller_count']

In [20]:
FEATURES = feat_base + feat_lags_txn + feat_rolls_txn + feat_lags_amt + feat_rolls_amt + feat_misc + [BILLER_COL]


In [21]:
# Fill missing values for features
train_X_all = train[FEATURES].fillna(-1)
val_X_all = val[FEATURES].fillna(-1)

In [22]:
# ---------- TRANSACTIONS MODEL: RANDOMIZED HYPERPARAM SEARCH ----------
print("\nTuning transactions model (poisson objective) with randomized search...")




Tuning transactions model (poisson objective) with randomized search...


In [23]:
dtrain = lgb.Dataset(train_X_all, label=train[TARGET_TXN].astype(float), categorical_feature=[BILLER_COL], free_raw_data=False)
dval = lgb.Dataset(val_X_all, label=val[TARGET_TXN].astype(float), reference=dtrain, categorical_feature=[BILLER_COL], free_raw_data=False)


In [24]:
# Parameter space to sample from
param_space = {
    'learning_rate': [0.01, 0.03, 0.05, 0.08, 0.1],
    'num_leaves': [31, 48, 64, 96, 128],
    'min_data_in_leaf': [20, 50, 100, 200],
    'feature_fraction': [0.6, 0.8, 1.0],
    'bagging_fraction': [0.6, 0.8, 1.0],
    'bagging_freq': [0, 1, 5],
    'lambda_l1': [0.0, 0.1, 0.5],
    'lambda_l2': [0.0, 0.1, 0.5]
}

fixed_params = {
    'objective': 'poisson',
    'metric': 'rmse',
    'verbosity': -1,
    'seed': random_seed
}

In [25]:
def sample_params(space):
    p = {k: random.choice(v) for k, v in space.items()}
    p.update(fixed_params)
    return p

In [26]:
best_score = float('inf')
best_params_txn = None
best_model_txn = None

In [ ]:
for t in range(N_TRIALS):
    p = sample_params(param_space)
    # reduce complexity slightly for quick trials
    model = lgb.train(p, dtrain, valid_sets=[dtrain, dval], num_boost_round=1000)
    preds = model.predict(val_X_all, num_iteration=model.best_iteration)
    preds = np.maximum(preds, 0)
    rmse = sqrt(mean_squared_error(val[TARGET_TXN].values, preds))
    if rmse < best_score:
        best_score = rmse
        best_params_txn = p
        best_model_txn = model
    if (t+1) % 5 == 0:
        print(f"Trial {t+1}/{N_TRIALS} | current best RMSE: {best_score:.2f}")

Trial 5/25 | current best RMSE: 13.00
Trial 10/25 | current best RMSE: 13.00
Trial 15/25 | current best RMSE: 13.00


In [ ]:
print("Best RMSE (transactions) on validation:", best_score)
print("Best params (txn):", best_params_txn)
# save params
with open(BEST_PARAMS_TXN, 'w') as f:
    json.dump(best_params_txn, f, indent=2)


In [ ]:
# Retrain final transactions model with best params (full early stopping and more rounds)
print("\nRetraining final transactions model with best params...")
final_params_txn = best_params_txn.copy()
final_model_txn = lgb.train(final_params_txn, dtrain, valid_sets=[dtrain, dval], num_boost_round=2000, early_stopping_rounds=100, verbose_eval=100)
final_model_txn.save_model(MODEL_TXN_PATH)
print("Saved transactions model to:", MODEL_TXN_PATH)

In [ ]:
# ---------- PREDICT TRANSACTIONS ON VAL (used for amount model features optionally) ----------
val_pred_txn = final_model_txn.predict(val_X_all, num_iteration=final_model_txn.best_iteration)
val_pred_txn = np.maximum(val_pred_txn, 0)

In [ ]:
# OPTIONALLY add predicted txn as a feature for amount model (helps)
train_X_all['pred_txn_placeholder'] = -1  # not used for train
val_X_all['pred_txn'] = val_pred_txn

In [ ]:
# ---------- AMOUNT MODEL: RANDOMIZED HYPERPARAM SEARCH (log1p target) ----------
print("\nTuning amount model (log1p regression) with randomized search...")


In [ ]:
# prepare labels (log1p)
y_train_amt_log = np.log1p(train[TARGET_AMT].astype(float))
y_val_amt_log = np.log1p(val[TARGET_AMT].astype(float))


In [ ]:
dtrain_amt = lgb.Dataset(train_X_all, label=y_train_amt_log, categorical_feature=[BILLER_COL], free_raw_data=False)
dval_amt = lgb.Dataset(val_X_all, label=y_val_amt_log, reference=dtrain_amt, categorical_feature=[BILLER_COL], free_raw_data=False)

In [ ]:
fixed_params_amt = {
    'objective': 'regression',
    'metric': 'rmse',
    'verbosity': -1,
    'seed': random_seed
}

In [ ]:
param_space_amt = {
    'learning_rate': [0.01, 0.03, 0.05, 0.08],
    'num_leaves': [31, 48, 64, 96],
    'min_data_in_leaf': [20, 50, 100, 200],
    'feature_fraction': [0.6, 0.8, 1.0],
    'bagging_fraction': [0.6, 0.8, 1.0],
    'bagging_freq': [0,1,5],
    'lambda_l1': [0.0, 0.1, 0.5],
    'lambda_l2': [0.0, 0.1, 0.5]
}

In [ ]:
best_score_amt = float('inf')
best_params_amt = None
best_model_amt = None

In [ ]:
def sample_params_amt(space):
    p = {k: random.choice(v) for k, v in space.items()}
    p.update(fixed_params_amt)
    return p

In [ ]:
for t in range(N_TRIALS):
    p = sample_params_amt(param_space_amt)
    model = lgb.train(p, dtrain_amt, valid_sets=[dtrain_amt, dval_amt], num_boost_round=1000, early_stopping_rounds=50, verbose_eval=False)
    preds_log = model.predict(val_X_all, num_iteration=model.best_iteration)
    preds = np.expm1(preds_log)
    rmse_amt = sqrt(mean_squared_error(val[TARGET_AMT].values, preds))
    if rmse_amt < best_score_amt:
        best_score_amt = rmse_amt
        best_params_amt = p
        best_model_amt = model
    if (t+1) % 5 == 0:
        print(f"Trial {t+1}/{N_TRIALS} | current best RMSE (amt): {best_score_amt:.2f}")

In [ ]:
print("Best RMSE (amount) on validation:", best_score_amt)
print("Best params (amount):", best_params_amt)
with open(BEST_PARAMS_AMT, 'w') as f:
    json.dump(best_params_amt, f, indent=2)

In [ ]:
# Retrain final amount model with best params
print("\nRetraining final amount model with best params...")
final_params_amt = best_params_amt.copy()
final_model_amt = lgb.train(final_params_amt, dtrain_amt, valid_sets=[dtrain_amt, dval_amt], num_boost_round=2000, early_stopping_rounds=100, verbose_eval=100)
final_model_amt.save_model(MODEL_AMT_PATH)
print("Saved amount model to:", MODEL_AMT_PATH)

In [ ]:
# ---------- VALIDATION PREDICTIONS & METRICS ----------
print("\nGenerating final predictions on validation set...")
# Predict transactions (already have val_pred_txn, but predict again from final_model_txn for safety)
pred_txn_final = final_model_txn.predict(val_X_all, num_iteration=final_model_txn.best_iteration)
pred_txn_final = np.maximum(pred_txn_final, 0)
pred_amt_log_final = final_model_amt.predict(val_X_all, num_iteration=final_model_amt.best_iteration)
pred_amt_final = np.expm1(pred_amt_log_final)
pred_amt_final = np.maximum(pred_amt_final, 0)


In [ ]:
# Metrics overall
metrics_txn = metrics_reg(val[TARGET_TXN].values, pred_txn_final)
metrics_amt = metrics_reg(val[TARGET_AMT].values, pred_amt_final)
print("Validation metrics - transactions:", metrics_txn)
print("Validation metrics - amount:", metrics_amt)


In [ ]:
# Build results DataFrame
results = val[[DATE_COL, BILLER_COL, TARGET_TXN, TARGET_AMT]].copy().reset_index(drop=True)
results = results.rename(columns={TARGET_TXN: 'actual_total_transactions', TARGET_AMT: 'actual_total_amount'})
results['pred_total_transactions'] = pred_txn_final
results['pred_total_amount'] = pred_amt_final
results['error_txn'] = results['actual_total_transactions'] - results['pred_total_transactions']
results['error_amt'] = results['actual_total_amount'] - results['pred_total_amount']

In [ ]:
results.to_csv(OUT_PRED_CSV, index=False)
print("Saved validation predictions to:", OUT_PRED_CSV)


In [ ]:
# ---------- PER-BILLER ERROR REPORT ----------
print("Computing per-biller metrics...")
grouped = results.groupby(BILLER_COL).apply(
    lambda g: pd.Series({
        'count': len(g),
        'MAE_txn': mean_absolute_error(g['actual_total_transactions'], g['pred_total_transactions']),
        'RMSE_txn': sqrt(mean_squared_error(g['actual_total_transactions'], g['pred_total_transactions'])),
        'MAPE_txn': safe_mape(g['actual_total_transactions'].values, g['pred_total_transactions'].values),
        'R2_txn': r2_score(g['actual_total_transactions'], g['pred_total_transactions']),
        'MAE_amt': mean_absolute_error(g['actual_total_amount'], g['pred_total_amount']),
        'RMSE_amt': sqrt(mean_squared_error(g['actual_total_amount'], g['pred_total_amount'])),
        'MAPE_amt': safe_mape(g['actual_total_amount'].values, g['pred_total_amount'].values),
        'R2_amt': r2_score(g['actual_total_amount'], g['pred_total_amount'])
    })
).reset_index()

In [ ]:
grouped = grouped.sort_values('MAPE_txn', ascending=False).reset_index(drop=True)
grouped.to_csv(OUT_BILLER_ERR, index=False)
print("Saved per-biller error report to:", OUT_BILLER_ERR)

In [ ]:
# Print top-5 worst & best billers by txn MAPE
print("\nTop 10 worst billers by txn MAPE:")
print(grouped[['biller_id','count','MAE_txn','MAPE_txn','RMSE_txn']].head(5))
print("\nTop 10 best billers by txn MAPE:")
print(grouped[['biller_id','count','MAE_txn','MAPE_txn','RMSE_txn']].tail(5))

In [ ]:
# ---------- PLOTS ----------
# Sample some billers to visualize (up to 3)
sample_billers = results[BILLER_COL].unique()[:3]


In [ ]:
# Actual vs Pred for sample billers (txn & amt)
for title, actual_col, pred_col in [
    ("Transactions", 'actual_total_transactions', 'pred_total_transactions'),
    ("Amount", 'actual_total_amount', 'pred_total_amount')
]:
    plt.figure(figsize=(10,4))
    for b in sample_billers:
        sub = results[results[BILLER_COL] == b].sort_values(DATE_COL)
        plt.plot(sub[DATE_COL], sub[actual_col], label=f"{b} actual")
        plt.plot(sub[DATE_COL], sub[pred_col], linestyle='--', label=f"{b} pred")
    plt.title(f"Actual vs Predicted - {title} (sample billers)")
    plt.xlabel("Date"); plt.ylabel(title)
    plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Residual histograms
plt.figure(figsize=(8,4)); plt.hist(results['error_txn'], bins=60); plt.title("Residuals - Transactions"); plt.xlabel("actual - pred"); plt.tight_layout(); plt.show()
plt.figure(figsize=(8,4)); plt.hist(results['error_amt'], bins=60); plt.title("Residuals - Amount"); plt.xlabel("actual - pred"); plt.tight_layout(); plt.show()

In [ ]:
# Feature importance
def show_fi(m, top_n=20, title="Feature importance"):
    fi = pd.DataFrame({'feature': m.feature_name(), 'importance': m.feature_importance(importance_type='gain')})
    fi = fi.sort_values('importance', ascending=False).reset_index(drop=True)
    print(title); print(fi.head(top_n))
    plt.figure(figsize=(8,6)); plt.barh(fi['feature'].head(top_n)[::-1], fi['importance'].head(top_n)[::-1]); plt.title(title);plt.tight_layout(); plt.show()

In [ ]:
show_fi(final_model_txn, top_n=20, title="Feature importance - Transactions")
show_fi(final_model_amt, top_n=20, title="Feature importance - Amount")

In [ ]:
print("\nAll done. Files saved:")
print(" - Validation predictions:", OUT_PRED_CSV)
print(" - Per-biller error report:", OUT_BILLER_ERR)
print(" - Best params transactions:", BEST_PARAMS_TXN)
print(" - Best params amount:", BEST_PARAMS_AMT)
print(" - Models:", MODEL_TXN_PATH, MODEL_AMT_PATH)